In [2]:
#!/usr/bin/env python3
"""
sector_diagnostico_tecnico.py

Calcula un diagnóstico sectorial basado en RSI, momentum y volumen
de los ETFs sectoriales. Compara con el estado macro vigente
para determinar coherencia.

CERO llamadas a la API — todo desde sector.sector_snapshot.

Output: sector.sector_diagnostico_tecnico (una fila por fecha)
"""

import os
import logging
import psycopg2
import psycopg2.extras
from datetime import datetime, date
from dotenv import load_dotenv

# ── ENV ────────────────────────────────────────────────────────────────────────
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD]):
    raise EnvironmentError("Faltan variables de entorno de PostgreSQL")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")

# ── Logging ────────────────────────────────────────────────────────────────────
LOG_DIR  = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/sector_diagnostico_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

# ── Clasificación de sectores ──────────────────────────────────────────────────
# ETFs sectoriales SPDR clasificados por comportamiento en el ciclo
DEFENSIVOS = ["XLV", "XLP", "XLU", "GLD", "TLT"]
CICLICOS   = ["XLK", "XLY", "XLF", "XLI", "XLE"]
MIXTOS     = ["XLB", "XLRE", "XLC"]

# ── DB ─────────────────────────────────────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD, host=POSTGRES_HOST, port=POSTGRES_PORT
    )


# ── Leer último snapshot sectorial ────────────────────────────────────────────
def leer_snapshot_sectorial(conn) -> dict:
    """
    Lee el último run de sector.sector_snapshot para los ETFs sectoriales.
    Devuelve dict con ticker como clave.
    """
    tickers = DEFENSIVOS + CICLICOS + MIXTOS
    placeholders = ",".join(["%s"] * len(tickers))

    sql = f"""
        SELECT
            ticker,
            rsi_rs_semanal,
            rsi_rs_diario,
            ret_3m,
            ret_1m,
            vol_ratio AS volume_ratio_20d,
            obv_slope,
            score_total,
            estado,
            alineacion_macro
        FROM sector.sector_snapshot
        WHERE ticker IN ({placeholders})
          AND run_id = (
              SELECT MAX(run_id)
              FROM sector.sector_snapshot
          )
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql, tickers)
        rows = cur.fetchall()

    return {r["ticker"]: dict(r) for r in rows}


# ── Leer estado macro vigente ──────────────────────────────────────────────────
def leer_estado_macro(conn) -> str:
    sql = """
        SELECT estado_macro
        FROM macro.macro_diagnostico
        ORDER BY calculado_en DESC
        LIMIT 1
    """
    with conn.cursor() as cur:
        cur.execute(sql)
        row = cur.fetchone()
        return row[0] if row else None


# ── Calcular score por grupo ───────────────────────────────────────────────────
def calcular_score_grupo(snapshot: dict, tickers: list) -> dict:
    """
    Calcula métricas promedio para un grupo de tickers.
    Ignora los que no tienen datos.
    """
    rsi_values  = []
    vol_values  = []
    ret_values  = []
    obv_values  = []

    for t in tickers:
        if t not in snapshot:
            continue
        d = snapshot[t]
        if d.get("rsi_rs_semanal") is not None:
            rsi_values.append(float(d["rsi_rs_semanal"]))
        if d.get("volume_ratio_20d") is not None:
            vol_values.append(float(d["volume_ratio_20d"]))
        if d.get("ret_3m") is not None:
            ret_values.append(float(d["ret_3m"]))
        if d.get("obv_slope") is not None:
            obv_values.append(float(d["obv_slope"]))

    def avg(lst):
        return round(sum(lst) / len(lst), 2) if lst else None

    return {
        "rsi_promedio":  avg(rsi_values),
        "vol_promedio":  avg(vol_values),
        "ret_promedio":  avg(ret_values),
        "obv_promedio":  avg(obv_values),
        "n_tickers":     len(rsi_values),
    }


# ── Top líderes y rezagados ───────────────────────────────────────────────────
def calcular_top_bottom(snapshot: dict, tickers: list) -> tuple[str, str]:
    """
    Devuelve (top_3_lideres, top_3_rezagados) ordenados por RSI semanal.
    """
    datos = []
    for t in tickers:
        if t in snapshot and snapshot[t].get("rsi_rs_semanal") is not None:
            datos.append((t, float(snapshot[t]["rsi_rs_semanal"])))

    datos.sort(key=lambda x: x[1], reverse=True)

    lideres  = ", ".join([t for t, _ in datos[:3]])
    rezagados = ", ".join([t for t, _ in datos[-3:]])

    return lideres, rezagados


# ── Diagnóstico sectorial ──────────────────────────────────────────────────────
def calcular_diagnostico(def_score: dict, cic_score: dict,
                          estado_macro: str) -> tuple[str, str, str]:
    """
    Compara defensivos vs cíclicos y determina:
    - diagnostico_sector
    - coherencia con estado_macro
    - nota explicativa
    """
    rsi_def = def_score.get("rsi_promedio") or 50
    rsi_cic = cic_score.get("rsi_promedio") or 50
    vol_def = def_score.get("vol_promedio") or 1.0
    vol_cic = cic_score.get("vol_promedio") or 1.0

    diferencia_rsi = rsi_def - rsi_cic      # positivo = defensivos lideran
    diferencia_vol = vol_def - vol_cic       # positivo = más volumen en defensivos

    # ── Diagnóstico sectorial
    if diferencia_rsi >= 15 and diferencia_vol >= 0.1:
        diagnostico = "CONFIRMA_SLOWDOWN"
        nota = (f"Defensivos lideran con RSI {rsi_def:.0f} vs cíclicos {rsi_cic:.0f}. "
                f"Volumen confirma la rotación hacia sectores defensivos.")

    elif diferencia_rsi >= 10:
        diagnostico = "CONFIRMA_SLOWDOWN"
        nota = (f"Defensivos lideran con RSI {rsi_def:.0f} vs cíclicos {rsi_cic:.0f}. "
                f"Volumen no confirma con fuerza.")

    elif diferencia_rsi <= -15 and diferencia_vol <= -0.1:
        diagnostico = "CONFIRMA_EXPANSION"
        nota = (f"Cíclicos lideran con RSI {rsi_cic:.0f} vs defensivos {rsi_def:.0f}. "
                f"Volumen confirma entrada de dinero en sectores de riesgo.")

    elif diferencia_rsi <= -10:
        diagnostico = "CONFIRMA_EXPANSION"
        nota = (f"Cíclicos lideran con RSI {rsi_cic:.0f} vs defensivos {rsi_def:.0f}. "
                f"Volumen no confirma con fuerza.")

    elif rsi_def >= 70 and rsi_cic >= 70:
        diagnostico = "CONFIRMA_CONTRACTION"
        nota = (f"Todos los sectores en RSI alto — mercado en pánico o máximos. "
                f"Defensivos RSI {rsi_def:.0f}, Cíclicos {rsi_cic:.0f}.")

    else:
        diagnostico = "SEÑAL_MIXTA"
        nota = (f"Sin dirección clara. Defensivos RSI {rsi_def:.0f}, "
                f"Cíclicos RSI {rsi_cic:.0f}. Diferencia insuficiente para señal.")

    # ── Coherencia con estado macro
    coherencia_map = {
        ("SLOWDOWN",    "CONFIRMA_SLOWDOWN"):    "ALTA",
        ("SLOWDOWN",    "SEÑAL_MIXTA"):          "MEDIA",
        ("SLOWDOWN",    "CONFIRMA_EXPANSION"):   "CONTRADICE",
        ("EXPANSION",   "CONFIRMA_EXPANSION"):   "ALTA",
        ("EXPANSION",   "SEÑAL_MIXTA"):          "MEDIA",
        ("EXPANSION",   "CONFIRMA_SLOWDOWN"):    "CONTRADICE",
        ("CONTRACTION", "CONFIRMA_CONTRACTION"): "ALTA",
        ("CONTRACTION", "CONFIRMA_SLOWDOWN"):    "MEDIA",
        ("CONTRACTION", "CONFIRMA_EXPANSION"):   "CONTRADICE",
        ("RECOVERY",    "CONFIRMA_EXPANSION"):   "ALTA",
        ("RECOVERY",    "SEÑAL_MIXTA"):          "MEDIA",
        ("RECOVERY",    "CONFIRMA_SLOWDOWN"):    "CONTRADICE",
    }
    coherencia = coherencia_map.get((estado_macro, diagnostico), "MEDIA")

    return diagnostico, coherencia, nota


# ── INSERT ─────────────────────────────────────────────────────────────────────
INSERT_SQL = """
INSERT INTO sector.sector_diagnostico_tecnico (
    fecha, run_id,
    score_defensivos, score_ciclicos, score_mixtos,
    top_3_lideres, top_3_rezagados,
    vol_defensivos, vol_ciclicos,
    diagnostico_sector, estado_macro, coherencia, nota
)
VALUES (
    %(fecha)s, %(run_id)s,
    %(score_defensivos)s, %(score_ciclicos)s, %(score_mixtos)s,
    %(top_3_lideres)s, %(top_3_rezagados)s,
    %(vol_defensivos)s, %(vol_ciclicos)s,
    %(diagnostico_sector)s, %(estado_macro)s, %(coherencia)s, %(nota)s
)
ON CONFLICT (fecha) DO UPDATE SET
    run_id             = EXCLUDED.run_id,
    score_defensivos   = EXCLUDED.score_defensivos,
    score_ciclicos     = EXCLUDED.score_ciclicos,
    score_mixtos       = EXCLUDED.score_mixtos,
    top_3_lideres      = EXCLUDED.top_3_lideres,
    top_3_rezagados    = EXCLUDED.top_3_rezagados,
    vol_defensivos     = EXCLUDED.vol_defensivos,
    vol_ciclicos       = EXCLUDED.vol_ciclicos,
    diagnostico_sector = EXCLUDED.diagnostico_sector,
    estado_macro       = EXCLUDED.estado_macro,
    coherencia         = EXCLUDED.coherencia,
    nota               = EXCLUDED.nota;
"""


# ── MAIN ───────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print(f"  SECTOR DIAGNÓSTICO TÉCNICO — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"  run_id: {RUN_ID}")
    print(f"{'='*65}\n")
    print("  Leyendo desde sector.sector_snapshot — sin API calls\n")

    conn = get_conn()

    # 1. Leer datos
    snapshot     = leer_snapshot_sectorial(conn)
    estado_macro = leer_estado_macro(conn)

    logging.info(f"Tickers en snapshot: {list(snapshot.keys())}")
    logging.info(f"Estado macro vigente: {estado_macro}")

    if not snapshot:
        print("  ✗ Sin datos en sector_snapshot. Corré sector_precios.py primero.")
        conn.close()
        return

    # 2. Calcular scores por grupo
    def_score = calcular_score_grupo(snapshot, DEFENSIVOS)
    cic_score = calcular_score_grupo(snapshot, CICLICOS)
    mix_score = calcular_score_grupo(snapshot, MIXTOS)

    # 3. Top líderes y rezagados (sobre todos los sectoriales)
    todos = DEFENSIVOS + CICLICOS + MIXTOS
    top_lideres, top_rezagados = calcular_top_bottom(snapshot, todos)

    # 4. Diagnóstico y coherencia
    diagnostico, coherencia, nota = calcular_diagnostico(
        def_score, cic_score, estado_macro
    )

    # 5. Mostrar en pantalla
    print(f"  Estado macro:      {estado_macro}")
    print(f"  RSI defensivos:    {def_score['rsi_promedio']} "
          f"(vol: {def_score['vol_promedio']})")
    print(f"  RSI cíclicos:      {cic_score['rsi_promedio']} "
          f"(vol: {cic_score['vol_promedio']})")
    print(f"  RSI mixtos:        {mix_score['rsi_promedio']}")
    print(f"  Top líderes:       {top_lideres}")
    print(f"  Top rezagados:     {top_rezagados}")
    print(f"\n  {'─'*55}")
    print(f"  Diagnóstico:       {diagnostico}")
    print(f"  Coherencia macro:  {coherencia}")
    print(f"  Nota:              {nota}")
    print(f"  {'─'*55}\n")

    # 6. Insertar en DB
    payload = {
        "fecha":             date.today(),
        "run_id":            RUN_ID,
        "score_defensivos":  def_score["rsi_promedio"],
        "score_ciclicos":    cic_score["rsi_promedio"],
        "score_mixtos":      mix_score["rsi_promedio"],
        "top_3_lideres":     top_lideres,
        "top_3_rezagados":   top_rezagados,
        "vol_defensivos":    def_score["vol_promedio"],
        "vol_ciclicos":      cic_score["vol_promedio"],
        "diagnostico_sector":diagnostico,
        "estado_macro":      estado_macro,
        "coherencia":        coherencia,
        "nota":              nota,
    }

    with conn.cursor() as cur:
        cur.execute(INSERT_SQL, payload)
    conn.commit()
    conn.close()

    print(f"  ✓ Diagnóstico guardado en sector.sector_diagnostico_tecnico")
    print(f"\n{'='*65}")
    print(f"  Pipeline completo — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()

2026-04-10 09:11:28,978 | INFO | Tickers en snapshot: ['GLD', 'TLT', 'XLC', 'XLP', 'XLY', 'XLE', 'XLF', 'XLI', 'XLB', 'XLRE', 'XLV', 'XLK', 'XLU']
2026-04-10 09:11:28,980 | INFO | Estado macro vigente: SLOWDOWN



  SECTOR DIAGNÓSTICO TÉCNICO — 2026-04-10 09:11
  run_id: 20260410_0911

  Leyendo desde sector.sector_snapshot — sin API calls

  Estado macro:      SLOWDOWN
  RSI defensivos:    60.3 (vol: 0.77)
  RSI cíclicos:      50.44 (vol: 0.76)
  RSI mixtos:        62.23
  Top líderes:       XLE, XLI, XLRE
  Top rezagados:     XLK, XLF, XLY

  ───────────────────────────────────────────────────────
  Diagnóstico:       SEÑAL_MIXTA
  Coherencia macro:  MEDIA
  Nota:              Sin dirección clara. Defensivos RSI 60, Cíclicos RSI 50. Diferencia insuficiente para señal.
  ───────────────────────────────────────────────────────

  ✓ Diagnóstico guardado en sector.sector_diagnostico_tecnico

  Pipeline completo — 09:11:28

